# Generic Exploratory Data Analysis (EDA) Template
### JPMorgan Corporate Finance & Market Data Project

**Instructions:** 
1. Change the `FILE_PATH` variable in the box below to point to your dataset (e.g. your Bloomberg spin-off CSV or WRDS output).
2. The notebook automatically detects if the file is a `.csv` or `.parquet` format and processes it accordingly.
3. Run all cells to generate structured data quality checks, statistical summaries, and exploratory visualizations.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting styles to look clean and professional
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# ==========================================================
# CONFIGURATION: CHANGE THIS TO YOUR FILE PATH
# Supports: 'your_file.csv' or 'your_file.parquet'
# ==========================================================
FILE_PATH = "spy_spinoff.csv" 

def load_data(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found at: {path}. Please check the path and try again.")
        
    ext = os.path.splitext(path)[-1].lower()
    if ext == '.csv':
        print(f"Loading CSV file from: {path}")
        return pd.read_csv(path)
    elif ext == '.parquet':
        print(f"Loading Parquet file from: {path}")
        return pd.read_parquet(path)
    else:
        raise ValueError("Unsupported file format! Please use a .csv or .parquet file.")

# Load the dataframe
df = load_data(FILE_PATH)
print(f"Data successfully loaded! Shape: {df.shape[0]} rows, {df.shape[1]} columns.")

## 1. High-Level Structural Audit
Inspect data types, column schemas, and sample the first few entries.

In [ ]:
print("--- DATAFRAME INFO ---")
print(df.info())

print("\n--- FIRST 5 ROWS ---")
df.head()

## 2. Data Quality & Completeness Audit
Check for structural gaps, missing values, and duplicate rows.

In [ ]:
print("--- MISSING VALUES AUDIT ---")
missing_count = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_table = pd.concat([missing_count, missing_pct], axis=1, keys=['Missing Count', 'Percentage (%)'])
missing_table = missing_table[missing_table['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False)

if missing_table.empty:
    print("Clean data! No missing values detected.")
else:
    print(missing_table)
    
print("\n--- DUPLICATE ROW CHECK ---")
duplicate_count = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count} ({duplicate_count/len(df)*100:.2f}%)")

## 3. Statistical Summaries
Segment columns into numerical vs. categorical variables and evaluate basic mathematical metrics.

In [ ]:
# Identify numerical vs categorical splits
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

print(f"Detected {len(num_cols)} numerical columns and {len(cat_cols)} categorical columns.")

if num_cols:
    print("\n--- NUMERICAL COLUMNS STATS ---")
    display(df[num_cols].describe().T)

if cat_cols:
    print("\n--- CATEGORICAL COLUMNS STATS ---")
    display(df[cat_cols].describe().T)

## 4. Distributions & Outliers
Automated visualization of value splits and outlying elements across the top numerical attributes.

In [ ]:
# Automatically plots distributions for up to the first 6 numerical columns to avoid cluttering memory
cols_to_plot = num_cols[:6]

if cols_to_plot:
    print(f"Visualizing distributions for: {cols_to_plot}")
    fig, axes = plt.subplots(len(cols_to_plot), 2, figsize=(14, 4 * len(cols_to_plot)))
    if len(cols_to_plot) == 1:
        axes = np.expand_dims(axes, axis=0)
        
    for i, col in enumerate(cols_to_plot):
        # Histogram/KDE
        sns.histplot(df[col], kde=True, ax=axes[i, 0], color='skyblue')
        axes[i, 0].set_title(f'{col} Distribution')
        
        # Boxplot for Outliers
        sns.boxplot(x=df[col], ax=axes[i, 1], color='salmon')
        axes[i, 1].set_title(f'{col} Outliers')
        
    plt.tight_layout()
    plt.show()
else:
    print("No numerical columns found to plot.")

## 5. Correlation Matrix
Evaluate co-movements and multi-collinearity trends across the loaded features.

In [ ]:
if len(num_cols) > 1:
    print("--- CORRELATION MATRIX ---")
    corr = df[num_cols].corr()
    
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap='coolwarm', center=0, square=True, linewidths=.5)
    plt.title("Numerical Features Correlation Heatmap")
    plt.show()
else:
    print("Not enough numerical columns to compute a correlation matrix.")